# Train YOLO11 on the e-waste dataset (Google Colab)

**e-waste AI material recovery** — object detection training notebook.

Run the cells **top to bottom**. Before you start:
1. On your laptop, run `python -m ewaste_research.cli.prepare_yolo_data` to build the `dataset/` folder.
2. Zip that folder into `dataset.zip` (right-click the `dataset` folder > Send to > Compressed folder).
3. In Colab, set the runtime to GPU: **Runtime > Change runtime type > T4 GPU**.

This notebook mounts Google Drive, installs Ultralytics, gets your dataset,
trains YOLO11, and downloads the best weights (`best.pt`).

## Step 1 — Check the GPU
Confirm Colab gave you a GPU. If this errors, set the runtime type to T4 GPU.

In [ ]:
# Print the GPU details Colab assigned to this session.
# If you see 'command not found' or no GPU, switch the runtime to T4 GPU.
!nvidia-smi

## Step 2 — Mount Google Drive (optional but recommended)
This lets you save trained weights to your Drive so they survive if the
Colab session disconnects.

In [ ]:
# Import Colab's drive helper.
from google.colab import drive

# Mount your Google Drive at /content/drive. A popup will ask you to authorize.
drive.mount('/content/drive')

# Make a folder in your Drive to store results (no error if it already exists).
import os
os.makedirs('/content/drive/MyDrive/ewaste_yolo', exist_ok=True)
print('Drive mounted. Results will be copied to /content/drive/MyDrive/ewaste_yolo')

## Step 3 — Install Ultralytics (YOLO11)
Ultralytics is the library that provides YOLO11 and the `model.train()` API.

In [ ]:
# Install the Ultralytics package (includes YOLO11). -q keeps the log short.
!pip install -q ultralytics

# Import YOLO and print the version to confirm the install worked.
from ultralytics import YOLO
import ultralytics
ultralytics.checks()  # prints versions and confirms the GPU is visible

## Step 4 — Upload your dataset.zip
Run this cell, then click **Choose Files** and pick the `dataset.zip` you made
on your laptop. (For big datasets it is faster to upload the zip to Drive
manually and skip this cell — see the commented alternative below.)

In [ ]:
# --- Option A: upload directly from your computer ---
from google.colab import files

# Opens a file picker in the browser. Select your dataset.zip.
uploaded = files.upload()

# Grab the name of the file you just uploaded.
zip_name = list(uploaded.keys())[0]
print('Uploaded:', zip_name)

# --- Option B (faster for large zips): use a copy already in your Drive ---
# zip_name = '/content/drive/MyDrive/ewaste_yolo/dataset.zip'

In [ ]:
# Unzip the dataset into /content/dataset. -o overwrites without prompting,
# -q stays quiet so the output is not thousands of file names.
!unzip -o -q "$zip_name" -d /content/

# Show what landed there so you can confirm the structure looks right.
!ls -R /content/dataset | head -n 40

## Step 5 — Fix the dataset.yaml path for Colab
Your local `dataset.yaml` has a Windows path inside it. We rewrite the `path:`
line so it points at the unzipped folder here in Colab.

In [ ]:
# Path to the dataset.yaml that was inside your zip.
yaml_path = '/content/dataset/dataset.yaml'

# Read the file, replace the 'path:' line, and write it back.
with open(yaml_path, 'r') as f:
    lines = f.readlines()

# Rebuild the file: every line stays the same except the one starting with 'path:'.
with open(yaml_path, 'w') as f:
    for line in lines:
        if line.strip().startswith('path:'):
            f.write('path: /content/dataset\n')
        else:
            f.write(line)

# Print the corrected file so you can eyeball it.
print(open(yaml_path).read())

## Step 6 — Train YOLO11
We start from `yolo11n.pt` (the small 'nano' model). For a **small e-waste
dataset with few images per class under real-world conditions**, the settings
below favour heavy data augmentation and patience-based early stopping to fight
overfitting. Training on a T4 GPU typically takes a few minutes to ~an hour
depending on how many images you have.

In [ ]:
# Load the pretrained YOLO11-nano model. Starting from pretrained weights
# (transfer learning) is essential when you only have a small dataset.
model = YOLO('yolo11n.pt')

# Train. Each argument is explained below.
results = model.train(
    data='/content/dataset/dataset.yaml',  # points at your classes + image folders
    epochs=150,            # max passes over the data; early stopping may end sooner
    patience=30,           # stop early if val score does not improve for 30 epochs
    imgsz=640,             # train at 640x640 (matches ewaste_research/cli/prepare_yolo_data.py)
    batch=16,              # images per step; lower to 8 if you hit out-of-memory
    optimizer='AdamW',     # stable optimizer that works well on small datasets
    lr0=0.001,             # starting learning rate
    cos_lr=True,           # smoothly decay the learning rate (cosine schedule)
    dropout=0.1,           # light dropout to reduce overfitting
    # --- Augmentation: makes a small dataset act bigger and more varied ---
    hsv_h=0.015,           # tiny hue jitter
    hsv_s=0.7,             # strong saturation jitter (lighting varies a lot)
    hsv_v=0.4,             # brightness jitter
    degrees=10.0,          # small rotations
    translate=0.1,         # shift the image around
    scale=0.5,             # zoom in/out
    fliplr=0.5,            # 50% chance horizontal flip
    mosaic=1.0,            # stitch 4 images together (great for small data)
    mixup=0.1,             # blend two images occasionally
    # --- Bookkeeping ---
    project='/content/runs',  # where outputs are written
    name='ewaste_yolo11',     # subfolder name for this run
    plots=True,               # save training curves and confusion matrix
)

## Step 7 — Look at the results
YOLO saves the best weights and a set of plots. Let's print where they are and
show the results image.

In [ ]:
# Folder Ultralytics used for this run.
run_dir = '/content/runs/ewaste_yolo11'

# The best-performing weights are saved here.
best_weights = run_dir + '/weights/best.pt'
print('Best weights:', best_weights)

# Show the summary plot of training metrics.
from IPython.display import Image, display
display(Image(filename=run_dir + '/results.png'))

## Step 8 — Save best.pt to Drive and download it
Copy the weights to your Drive (backup) and trigger a browser download.

In [ ]:
import shutil

# Copy best.pt into your Drive folder so it is safe even if Colab disconnects.
shutil.copy(best_weights, '/content/drive/MyDrive/ewaste_yolo/best.pt')
print('Copied best.pt to Google Drive.')

# Also download it straight to your computer.
from google.colab import files
files.download(best_weights)

## Done!
Move the downloaded **best.pt** into your project at:

```
artifacts/models/best.pt
```

Then on your laptop run:

```powershell
python -m ewaste_research.cli.run_detector --model artifacts/models/best.pt --source 0
```

and the live window will use your trained e-waste detector.